In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot  as plt
import dunestyle.matplotlib as dunestyle


In [ ]:
data_dir = Path("RateAnalysis_data")
reference_file = data_dir / "reference_run_043387.npz"


In [ ]:
data = np.load(reference_file)

print(data.files)

In [ ]:
def load_npz(file):
    with np.load(file, allow_pickle=False) as data:
        channels = data["channels"]
        bin_edges = data["amplitude_bin_edges_adc"]
        try:
            histogram_rate_hz = data["histogram_rate_hz"]
            frequencies_khz = data["frequency_khz"]
            return channels, bin_edges,histogram_rate_hz,frequencies_khz
        except KeyError:
            histogram_before_rate_hz =  data["histogram_before_rate_hz"]
            histogram_after_rate_hz =  data["histogram_after_rate_hz"]
            frequencies_before_khz =  data["frequency_before_khz"]
            frequencies_after_khz =  data["frequency_after_khz"]
            return channels, bin_edges,histogram_before_rate_hz,histogram_after_rate_hz,frequencies_before_khz,frequencies_after_khz


In [ ]:
channel_to_plot = 2080

equalized_files = sorted(data_dir.glob("equalized_run_*.npz"))
if not equalized_files:
    raise FileNotFoundError(f"No equalized_run_*.npz files found in {data_dir}")

output_dir = data_dir / "plots" / f"channel_{channel_to_plot}"
output_dir.mkdir(parents=True, exist_ok=True)

chs_ref, bin_ref, hist_ref, freq_ref = load_npz(reference_file)


def find_channel_index(channels, channel, dataset_name):
    matches = np.flatnonzero(channels == channel)
    if matches.size == 0:
        raise ValueError(f"Channel {channel} is not present in {dataset_name}")
    return matches[0]


index_ref = find_channel_index(chs_ref, channel_to_plot, "reference data")
with np.load(reference_file, allow_pickle=False) as data:
    reference_run = str(data["run"].item())

plotted_runs = []
for equalized in equalized_files:
    (
        chs_eq,
        bin_eq,
        hist_before_eq,
        hist_after_eq,
        freq_before_eq,
        freq_after_eq,
    ) = load_npz(equalized)

    if channel_to_plot not in chs_eq:
        print(f"Skipping {equalized.name}: channel {channel_to_plot} is absent")
        continue
    index_eq = find_channel_index(chs_eq, channel_to_plot, equalized.name)

    with np.load(equalized, allow_pickle=False) as data:
        equalized_run = str(data["run"].item())
        threshold_adc = float(data["threshold_adc"][index_eq])
        target_rate_khz = float(data["target_frequency_khz"][index_eq])
        signal_start, signal_stop = data["signal_range"]

    if np.isfinite(threshold_adc):
        print(
            f"Run {equalized_run} | Channel {channel_to_plot}: "
            f"apply amplitude >= {threshold_adc:.2f} ADC "
            f"to match target {target_rate_khz:.3f} kHz "
            f"(achieved {freq_after_eq[index_eq]:.3f} kHz)"
        )
    else:
        print(f"Run {equalized_run}: rate-matching threshold unavailable")

    output_file = output_dir / (
        f"reference_histogram_run_{equalized_run}_channel_{channel_to_plot}.png"
    )

    fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

    ax.stairs(
        hist_ref[index_ref],
        bin_ref,
        linewidth=2,
        alpha=0.75,
        label=(
            f"Reference run {reference_run} | Channel {channel_to_plot} | "
            f"rate = {freq_ref[index_ref]:.3f} kHz"
        ),
    )

    ax.stairs(
        hist_before_eq[index_eq],
        bin_eq,
        linewidth=2,
        alpha=0.75,
        label=(
            f"Run {equalized_run} before equalization | "
            f"rate = {freq_before_eq[index_eq]:.3f} kHz"
        ),
    )

    ax.stairs(
        hist_after_eq[index_eq],
        bin_eq,
        linewidth=2,
        alpha=0.75,
        label=(
            f"Run {equalized_run} after equalization | "
            f"rate = {freq_after_eq[index_eq]:.3f} kHz"
        ),
    )
    if np.isfinite(threshold_adc):
        ax.axvline(
            threshold_adc,
            color="black",
            linestyle="--",
            linewidth=1.5,
            label=(
                f"Rate-matching cut: amplitude >= {threshold_adc:.2f} ADC "
                f"(target {target_rate_khz:.3f} kHz)"
            ),
        )
    else:
        ax.text(
            0.98, 0.75, "Rate-matching threshold unavailable",
            transform=ax.transAxes, ha="right", va="top",
        )
    ax.set_xlim(-100,12000)

    ax.set_xlabel(
        f"Baseline-to-peak amplitude, samples {signal_start}:1023 [ADC]"
    )
    ax.set_ylabel("Rate [Hz]")
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        output_file,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    plotted_runs.append(equalized_run)

print(f"Plotted {len(plotted_runs)} of {len(equalized_files)} runs; saved figures to {output_dir}")
